# VERA — RL Fine-Tuning (REINFORCE)

Runs **Phase 2 RL training** starting from an SFT checkpoint, using `rl_trainer_vera.py`.

**Loss**: REINFORCE + value baseline + entropy bonus + KL penalty vs frozen BC anchor  
**Expected time**: ~2–4 hrs for 100 epochs on T4 GPU (Language-Table)

---

## Before running
1. `Runtime → Change runtime type → T4 GPU`
2. Upload your SFT checkpoint (`best_sft_vera.pt`) to Drive at:
   ```
   MyDrive/VERA_LT_Checkpoints/rl_seed123/best_sft_vera.pt
   ```
   *(rename the file from your seed123 SFT run — it goes in the **rl_seed123** folder)*
3. Run all cells top-to-bottom

## Output
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/best_rl_vera.pt` — best checkpoint by return
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/rl_vera_log.json` — full epoch log
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/sample_efficiency.csv` — for plotting
- Live epoch-by-epoch output in cell 6 below

## Running multiple seeds
Open separate Colab tabs and change `SEED` in Cell 4 (e.g. `SEED = 42`, `SEED = 456`).  
Each seed needs its own SFT checkpoint folder:  
`rl_seed42/best_sft_vera.pt`, `rl_seed456/best_sft_vera.pt`

In [1]:
# ── Cell 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [2]:
# ── Cell 2: Clone repo (or use Drive copy) ─────────────────────────────────
import os, sys

REPO_DRIVE = '/content/drive/MyDrive/VLA-Robot-Learning'
REPO_LOCAL = '/content/VLA-Robot-Learning'

if os.path.isdir(REPO_DRIVE):
    print(f'Repo found on Drive at {REPO_DRIVE}')
    REPO = REPO_DRIVE
elif os.path.isdir(REPO_LOCAL):
    print(f'Repo already cloned at {REPO_LOCAL}')
    REPO = REPO_LOCAL
else:
    print('Cloning repo from GitHub...')
    os.system('git clone https://github.com/YOUR/VLA-Robot-Learning /content/VLA-Robot-Learning')
    REPO = REPO_LOCAL
    print(f'Cloned to {REPO}')

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print(f'Using repo: {REPO}')

Cloning repo from GitHub...
Cloned to /content/VLA-Robot-Learning
Using repo: /content/VLA-Robot-Learning


In [3]:
# ── Cell 3: Install dependencies ───────────────────────────────────────────
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip('ftfy regex tqdm')
pip('git+https://github.com/openai/CLIP.git')
pip('language-table')
pip('pyyaml')

print('Dependencies installed.')

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'ftfy regex tqdm']' returned non-zero exit status 1.

In [ ]:
# ── Cell 4: ⚙️  USER CONFIG — edit these values ────────────────────────────
import os

MYDRIVE    = '/content/drive/MyDrive'
CKPT_ROOT  = f'{MYDRIVE}/VERA_LT_Checkpoints'

# ── Which seed to run ──────────────────────────────────────────────────────
SEED = 123          # ← CHANGE to 42 or 456 in other Colab tabs for parallel runs
                    # Seed 123 recommended first (highest SFT baseline: ~10% SR)

# ── Path to your Language-Table data ──────────────────────────────────────
LT_DATA_PATH = f'{MYDRIVE}/language_table_data'   # ← edit if your data is elsewhere
                                                   # set to None to use synthetic data (for testing only)

# ── RL hyperparameters (recommended for highest task SR) ──────────────────
RL_EPOCHS          = 100       # total RL epochs (each epoch = num_rollouts episodes)
RL_NUM_ROLLOUTS    = 8         # episodes per gradient update (more = stabler at low SR)
RL_MAX_EP_STEPS    = 60        # match eval conditions
RL_LR              = 3e-5      # 3x default — DAgger logs showed 3e-5 fine for encoder
RL_ENTROPY_COEF    = 0.05      # 5x default — forces exploration (prevents mode collapse)
RL_KL_COEF         = 0.20      # 2x default — stronger BC anchor prevents forgetting
RL_VF_COEF         = 0.5       # value function weight (keep default)
RL_GAMMA           = 0.99      # discount factor
RL_GRAD_CLIP       = 1.0

# ── Visual encoder: keep frozen (DAgger showed unfreezing = noise at this scale) ──
FREEZE_CLIP          = True
UNFREEZE_CLIP_VISION = False

# ── Derived paths (don't edit) ─────────────────────────────────────────────
OUT_DIR = f'{CKPT_ROOT}/rl_seed{SEED}'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f'{OUT_DIR}/rl', exist_ok=True)

SFT_CKPT = f'{OUT_DIR}/best_sft_vera.pt'

print(f'Seed:         {SEED}')
print(f'Output dir:   {OUT_DIR}')
print(f'SFT checkpoint expected at: {SFT_CKPT}')

import os.path
if os.path.exists(SFT_CKPT):
    size_mb = os.path.getsize(SFT_CKPT) / 1e6
    print(f'  ✓ Found ({size_mb:.1f} MB) — ready to run RL')
else:
    print('  ✗ NOT FOUND — upload best_sft_vera.pt to this folder before running Cell 6!')

In [ ]:
# ── Cell 5: Write config.yaml with RL settings ─────────────────────────────
import yaml, os

CONFIG_PATH = f'{REPO}/configs/config.yaml'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Patch RL section
cfg['rl']['epochs']           = RL_EPOCHS
cfg['rl']['num_rollouts']     = RL_NUM_ROLLOUTS
cfg['rl']['max_episode_steps']= RL_MAX_EP_STEPS
cfg['rl']['lr']               = float(RL_LR)
cfg['rl']['entropy_coef']     = float(RL_ENTROPY_COEF)
cfg['rl']['kl_coef']          = float(RL_KL_COEF)
cfg['rl']['vf_coef']          = float(RL_VF_COEF)
cfg['rl']['gamma']            = float(RL_GAMMA)
cfg['rl']['grad_clip']        = float(RL_GRAD_CLIP)

# Patch model / training
cfg['model']['freeze_clip']           = FREEZE_CLIP
cfg['model']['unfreeze_clip_vision']  = UNFREEZE_CLIP_VISION
cfg['training']['seed']               = SEED
cfg['training']['output_dir']         = OUT_DIR

# Patch data path
if LT_DATA_PATH and os.path.isdir(LT_DATA_PATH):
    cfg['data']['episodes_path'] = LT_DATA_PATH
    cfg['data']['dataset_type']  = 'language_table'
    print(f'Data: Language-Table at {LT_DATA_PATH}')
else:
    cfg['data']['episodes_path'] = None
    cfg['data']['dataset_type']  = 'pkl'
    print('Data: synthetic (no real data found — results will be meaningless for SR)')

# Write patched config to a temp file (don't overwrite original)
RL_CONFIG_PATH = f'{OUT_DIR}/rl_config.yaml'
with open(RL_CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f)

print(f'Config written to: {RL_CONFIG_PATH}')
print()
print('RL settings:')
for k, v in cfg['rl'].items():
    print(f'  {k}: {v}')

In [ ]:
# ── Cell 6: Run RL training ────────────────────────────────────────────────
# Output streams live — each epoch prints:
#   RL Epoch  N | steps XXXXX | return X.XXXX | success XX.X% | policy X.XXXX | entropy X.XXXX | kl X.XXXX
#
# What to watch:
#   success XX.X%  → task success rate (this is what you want to go up)
#   entropy X.XXXX → should stay > 0.5 (if collapses to ~0, raise RL_ENTROPY_COEF)
#   kl X.XXXX      → should stay < 0.5 (if explodes, raise RL_KL_COEF)
#   policy X.XXXX  → can be negative (normal for policy gradient)
#
# Checkpoints auto-saved to: {OUT_DIR}/rl/best_rl_vera.pt

import subprocess, sys, os

if not os.path.exists(SFT_CKPT):
    raise FileNotFoundError(
        f'SFT checkpoint not found at {SFT_CKPT}\n'
        'Upload best_sft_vera.pt from your seed123 SFT run to that path first.'
    )

cmd = [
    sys.executable, '-m', 'training.rl_trainer_vera',
    '--config', RL_CONFIG_PATH,
]

print(f'Starting RL training (seed={SEED}, {RL_EPOCHS} epochs × {RL_NUM_ROLLOUTS} rollouts)...')
print(f'Command: {" ".join(cmd)}')
print('─' * 70)
print('Epoch | Steps   | Return | Success% | Policy  | Entropy | KL')
print('─' * 70)

# Stream output live
proc = subprocess.Popen(
    cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print('\n' + '─' * 70)
    print('✓ RL training complete!')
    print(f'  Best checkpoint: {OUT_DIR}/rl/best_rl_vera.pt')
    print(f'  Full log:        {OUT_DIR}/rl/rl_vera_log.json')
    print(f'  CSV for plots:   {OUT_DIR}/rl/sample_efficiency.csv')
else:
    print(f'\n✗ RL training exited with code {proc.returncode}')
    print('Scroll up for error details.')

In [ ]:
# ── Cell 7: Plot results ───────────────────────────────────────────────────
import json, os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

LOG_PATH = f'{OUT_DIR}/rl/rl_vera_log.json'

if not os.path.exists(LOG_PATH):
    print(f'Log not found at {LOG_PATH} — run Cell 6 first.')
else:
    with open(LOG_PATH) as f:
        log = json.load(f)

    epochs   = [r['epoch']        for r in log]
    sr       = [r['success_rate'] * 100 for r in log]
    ret      = [r['mean_return']  for r in log]
    entropy  = [r['entropy']      for r in log]
    kl       = [r['kl_loss']      for r in log]
    steps    = [r['cumulative_steps'] for r in log]

    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.suptitle(f'VERA RL Training — seed {SEED}', fontsize=13, fontweight='bold')

    # ── Success Rate ──
    ax = axes[0, 0]
    ax.plot(epochs, sr, color='#2563eb', linewidth=1.8)
    ax.axhline(10, color='#9ca3af', linewidth=1, linestyle='--', label='SFT baseline (~10%)')
    ax.fill_between(epochs, sr, 10, where=[s > 10 for s in sr],
                    color='#2563eb', alpha=0.12)
    ax.set_title('Task Success Rate (%)', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Success Rate (%)')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    ax.grid(True, alpha=0.3)

    # ── Mean Return ──
    ax = axes[0, 1]
    ax.plot(epochs, ret, color='#16a34a', linewidth=1.8)
    ax.set_title('Mean Episode Return', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Return')
    ax.grid(True, alpha=0.3)

    # ── Entropy ──
    ax = axes[1, 0]
    ax.plot(epochs, entropy, color='#d97706', linewidth=1.8)
    ax.axhline(0.5, color='#ef4444', linewidth=1, linestyle='--', label='Collapse risk < 0.5')
    ax.set_title('Policy Entropy (exploration)', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Entropy')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── KL vs BC ──
    ax = axes[1, 1]
    ax.plot(epochs, kl, color='#7c3aed', linewidth=1.8)
    ax.axhline(0.5, color='#ef4444', linewidth=1, linestyle='--', label='Forgetting risk > 0.5')
    ax.set_title('KL Divergence vs SFT Anchor', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('KL Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    PLOT_PATH = f'{OUT_DIR}/rl/training_curves.png'
    plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved to {PLOT_PATH}')

    best_sr  = max(sr)
    best_ep  = epochs[sr.index(best_sr)]
    final_sr = sr[-1]
    print(f'\nSummary (seed {SEED}):')
    print(f'  SFT baseline:  ~10%')
    print(f'  Best RL SR:    {best_sr:.1f}% (epoch {best_ep})')
    print(f'  Final RL SR:   {final_sr:.1f}% (epoch {epochs[-1]})')
    print(f'  Gain over SFT: +{best_sr - 10:.1f} pp')

## Troubleshooting

| Symptom | Fix |
|---|---|
| `FileNotFoundError: best_sft_vera.pt` | Upload seed123's SFT checkpoint to `VERA_LT_Checkpoints/rl_seed123/best_sft_vera.pt` |
| Success rate stays at 0% for 30+ epochs | Raise `RL_ENTROPY_COEF` to `0.08` in Cell 4, re-run Cell 5+6 |
| Entropy collapses to ~0 | Same fix — more entropy bonus |
| KL explodes > 1.0 | Raise `RL_KL_COEF` to `0.4` |
| CUDA out of memory | Lower `RL_NUM_ROLLOUTS` to `4` |
| Runtime disconnects | Checkpoint auto-saved every epoch — re-run Cell 6, training restarts from scratch but best_rl_vera.pt is safe on Drive |
| `ModuleNotFoundError: language_table` | Cell 3 didn't finish — re-run it |

## Running all 3 seeds in parallel
1. Open 3 separate Colab tabs (File → Open in new tab)
2. In each tab, change `SEED` in Cell 4 to `42`, `123`, `456`
3. Make sure each has its own SFT checkpoint:
   - `rl_seed42/best_sft_vera.pt` (from your seed42 SFT run)
   - `rl_seed123/best_sft_vera.pt` ← **start here first**
   - `rl_seed456/best_sft_vera.pt`